In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

# ---------------- CONFIG ----------------
BRONZE_JIRA  = "workspace.slainte_bronze.jira_issues_bronze"
DIM_EMPLOYEE = "workspace.slainte_gold.dim_employee"
DIM_TEAM     = "workspace.slainte_gold.dim_team"
DIM_PRIORITY = "workspace.slainte_gold.dim_priority"
DIM_STATUS   = "workspace.slainte_gold.dim_status"
DIM_TT       = "workspace.slainte_gold.dim_ticket_type"
DIM_SLA      = "workspace.slainte_gold.dim_sla"
DIM_PENALTY  = "workspace.slainte_gold.dim_penalty"
GOLD_DB      = "workspace.slainte_gold"
FACT_TABLE   = f"{GOLD_DB}.fact_ticket_perf"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

# ---------------- HELPERS ----------------
def safe_ts(col):
    return F.to_timestamp(
        F.when(
            (F.col(col).isNull()) | (F.lower(F.col(col)) == "none") |
            (F.lower(F.col(col)) == "null") | (F.col(col) == ""), None
        ).otherwise(F.col(col))
    )

def safe_date(col):
    return F.to_date(
        F.when(
            (F.col(col).isNull()) | (F.lower(F.col(col)) == "none") |
            (F.lower(F.col(col)) == "null") | (F.col(col) == ""), None
        ).otherwise(F.col(col))
    )

def safe_long(col):
    return F.when(
        (F.col(col).isNull()) | (F.lower(F.col(col)) == "none") |
        (F.lower(F.col(col)) == "null") | (F.col(col) == ""), None
    ).otherwise(F.col(col).cast("long"))

def safe_bool(col):
    return F.when(F.lower(F.col(col)) == "true", True)\
            .when(F.lower(F.col(col)) == "false", False)\
            .otherwise(None)

# ---------------- 1️⃣ DETECT LEVELS ----------------
jira_levels = spark.table(BRONZE_JIRA)\
    .filter((F.col("priority_label").isNotNull()) & (F.col("priority_label") != 99))\
    .select("priority_label").distinct().count()

sla_levels = spark.table(DIM_SLA)\
    .filter((F.col("priority_label").isNotNull()) & (F.col("priority_label") != 99))\
    .select("priority_label").distinct().count()

print(f"Jira Levels: {jira_levels}")
print(f"SLA Levels:  {sla_levels}")

# ---------------- 2️⃣ READ BRONZE ----------------
b = (
    spark.table(BRONZE_JIRA)
    .withColumn("source_user_id",   F.trim(F.col("user_id")))
    .withColumn("project",          F.trim(F.col("project_key")))
    .withColumn("ticket_number",    F.trim(F.col("ticket_number")))
    .withColumn("assignee",         F.trim(F.col("assignee")))
    .withColumn("team_key",         F.trim(F.element_at(F.split(F.coalesce(F.col("support_group"), F.lit("")), ","), 1)))
    .withColumn("ticket_type_key",  F.upper(F.trim(F.coalesce(F.col("ticket_type"), F.lit("")))))
    .withColumn("status_key",       F.upper(F.trim(F.coalesce(F.col("status"), F.lit("")))))
    .withColumn("priority_label",   F.col("priority_label").cast("int"))
    .withColumn(
        "priority_sla_label",
        F.when(F.lit(jira_levels) == F.lit(sla_levels), F.col("priority_label"))
         .when(F.lit(sla_levels) == 3,
               F.when(F.col("priority_label") <= 2, 1)
                .when(F.col("priority_label") == 3, 2)
                .otherwise(3))
         .otherwise(F.col("priority_label"))
    )
    .withColumn("created_at",               safe_ts("created_at"))
    .withColumn("resolved_at",              safe_ts("resolved_at"))
    .withColumn("last_update",              safe_ts("last_update"))
    .withColumn("due_date",                 safe_date("due_date"))
    .withColumn("first_response_time_ms",   safe_long("first_response_time_ms"))
    .withColumn("first_response_breached",  safe_bool("first_response_breached"))
    .withColumn("resolution_time_ms",       safe_long("resolution_time_ms"))
    .withColumn("resolution_breached",      safe_bool("resolution_breached"))
    .withColumn("ingestion_ts",             safe_ts("ingestion_timestamp"))
)

# ---------------- 3️⃣ DEDUP BRONZE ----------------
b = (
    b.withColumn("_rn", F.row_number().over(
        Window.partitionBy("source_user_id", "project", "ticket_number")
        .orderBy(F.col("last_update").desc_nulls_last())
    ))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .alias("b")
)

# ---------------- 4️⃣ DIMS ----------------
e   = spark.table(DIM_EMPLOYEE).alias("e")
t   = spark.table(DIM_TEAM).alias("t")
pr  = spark.table(DIM_PRIORITY).alias("pr")
st  = spark.table(DIM_STATUS).alias("st")
tt  = spark.table(DIM_TT).alias("tt")
s   = spark.table(DIM_SLA).alias("s")
pen = spark.table(DIM_PENALTY).alias("pen")

# ---------------- 5️⃣ JOINS ----------------
j = (
    b
    .join(e,   F.col("b.assignee")          == F.col("e.employee_name"),              "left")
    .join(t,   F.col("b.team_key")           == F.col("t.team_name"),                  "left")
    .join(pr,  F.col("b.priority_label")     == F.col("pr.priority_label"),            "left")
    .join(st,  F.col("b.status_key")         == F.upper(F.col("st.status_code")),      "left")
    .join(tt,  F.col("b.ticket_type_key")    == F.upper(F.col("tt.ticket_type_name")), "left")
    .join(s,   F.col("b.priority_sla_label") == F.col("s.priority_label"),             "left")
    # penalty: match on user + project + priority label
    # join twice — once for LATE_RESPONSE, once for LATE_RESOLUTION
    .join(
        pen.filter(F.col("pen.penalty_code") == "LATE_RESPONSE").alias("pen_resp"),
        (F.col("b.source_user_id")   == F.col("pen_resp.source_user_id")) &
        (F.col("b.project")          == F.col("pen_resp.project"))        &
        (F.col("b.priority_label")   == F.col("pen_resp.priority_group_label")),
        "left"
    )
    .join(
        pen.filter(F.col("pen.penalty_code") == "LATE_RESOLUTION").alias("pen_resol"),
        (F.col("b.source_user_id")   == F.col("pen_resol.source_user_id")) &
        (F.col("b.project")          == F.col("pen_resol.project"))        &
        (F.col("b.priority_label")   == F.col("pen_resol.priority_group_label")),
        "left"
    )
)

# ---------------- 6️⃣ PENALTY CALCULATION ----------------
j = (
    j
    # response penalty: applied if first_response_breached = true
    .withColumn(
        "response_penalty_amount",
        F.when(
            F.col("b.first_response_breached") == True,
            F.col("pen_resp.penalty_value")
        ).otherwise(F.lit(0.0))
    )
    .withColumn("response_penalty_unit", F.col("pen_resp.penalty_unit"))

    # resolution penalty: applied if resolution_breached = true
    .withColumn(
        "resolution_penalty_amount",
        F.when(
            F.col("b.resolution_breached") == True,
            F.col("pen_resol.penalty_value")
        ).otherwise(F.lit(0.0))
    )
    .withColumn("resolution_penalty_unit", F.col("pen_resol.penalty_unit"))

    # total penalty
    .withColumn(
        "total_penalty_amount",
        F.coalesce(F.col("response_penalty_amount"), F.lit(0.0)) +
        F.coalesce(F.col("resolution_penalty_amount"), F.lit(0.0))
    )
)

# ---------------- 7️⃣ FINAL SELECT ----------------
df_fact = j.select(
    F.col("b.ticket_number"),
    F.col("b.source_user_id"),
    F.col("b.project"),
    F.col("e.employee_id"),
    F.col("t.team_id"),
    F.col("tt.ticket_type_id"),
    F.col("pr.priority_id"),
    F.col("st.status_id"),
    F.col("s.sla_id"),
    F.col("pen_resp.penalty_id").alias("penalty_response_id"),
    F.col("pen_resol.penalty_id").alias("penalty_resolution_id"),
    F.col("b.created_at"),
    F.col("b.resolved_at"),
    F.col("b.last_update"),
    F.col("b.due_date"),
    F.col("b.first_response_time_ms"),
    F.col("b.first_response_breached"),
    F.col("b.resolution_time_ms"),
    F.col("b.resolution_breached"),
    F.col("response_penalty_amount"),
    F.col("response_penalty_unit"),
    F.col("resolution_penalty_amount"),
    F.col("resolution_penalty_unit"),
    F.col("total_penalty_amount"),
    F.col("b.ingestion_ts")
)

# ---------------- 8️⃣ FINAL DEDUP ----------------
df_fact = (
    df_fact.withColumn(
        "_rn",
        F.row_number().over(
            Window.partitionBy("ticket_number", "source_user_id", "project")
            .orderBy(F.col("last_update").desc_nulls_last(), F.col("ingestion_ts").desc_nulls_last())
        )
    )
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# ---------------- 9️⃣ SCHEMA ALIGNMENT ----------------
if spark.catalog.tableExists(FACT_TABLE):
    existing_cols = set(spark.table(FACT_TABLE).columns)
    for col in existing_cols - set(df_fact.columns):
        df_fact = df_fact.withColumn(col, F.lit(None))

# ---------------- 🔟 MERGE ----------------
merge_condition = """
    t.ticket_number  = s.ticket_number  AND
    t.source_user_id = s.source_user_id AND
    t.project        = s.project
"""

if not spark.catalog.tableExists(FACT_TABLE):
    df_fact.write.format("delta").mode("overwrite").saveAsTable(FACT_TABLE)
    print("✅ FACT CREATED")
else:
    (
        DeltaTable.forName(spark, FACT_TABLE).alias("t")
        .merge(df_fact.alias("s"), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("✅ FACT MERGED")